# Simulation for the probability of sites in the clonal frame

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
from tqdm import tqdm

In [2]:
def aff_P1(delta, L):
    return delta / (delta + L - 1)


def aff_P2(delta, L):
    return delta * (1 - (1 - 1/delta)**L) / L


def CF_P(N, rho, aff_P):
    return math.gamma(rho * aff_P + 1) * math.gamma(N) / math.gamma(rho * aff_P + N)

In [27]:
def birth_death_recomb_sim(N, rho, delta, L):
    """
    Simulate recombination events based on a birth-death process

    Simulate recombinations to compute the probability of a single site in a clona frame.

    param:
    N: A single integer for initial value - the number of lineages.
    rho: The birth rate - the recombination rate.
    delta: The mean value of the recombine segment.
    L: The number of sites.
    return:
    Hitting time, number of recombinations, number of final lineages,
    proportion of sites that are in the clonal frame.
    """
    if not isinstance(N, (int, np.integer)):
        raise ValueError(f"`N` must be a single integer!")
    if not isinstance(L, (int, np.integer)):
        raise ValueError(f"`L` must be a single integer!")

    t = 0.0
    k = N
    clonal_k = N
    non_clonal_k = 0
    recomb_num = 0
    in_CF = np.ones(L, dtype=bool)
    
    while k > 1:
        rate = k * (k - 1 + rho) / 2
        t += np.random.exponential(scale=1/rate)

        if np.random.random() < (k - 1) / (k - 1 + rho): # coalescence event
            prob_two_clonal = clonal_k * (clonal_k - 1) / 2
            prob_one_clonal = clonal_k * non_clonal_k
            prob_two_non_clonal = non_clonal_k * (non_clonal_k - 1) / 2
            prob = np.array([prob_two_clonal, prob_one_clonal, prob_two_non_clonal])
            choice = np.random.choice(3, p=prob / np.sum(prob))
            if choice == 0:
                clonal_k -= 1
            elif choice == 1:
                non_clonal_k -= 1
            else:
                non_clonal_k -= 1
            k -= 1

        else:                                            # recombination event
            recomb_num += 1
            start_pos = np.random.randint(0, L)
            recomb_length = np.random.geometric(p=1/delta)
            end_pos = start_pos + recomb_length
            end_pos = end_pos % L

            prob_on_clonal = clonal_k / k
            if np.random.random() < prob_on_clonal:
                if recomb_length >= L: # All sites are from donor, so no sites are in the clonal frame.
                    in_CF[:] = False
                else:
                    not_in_recomb = np.ones(L, dtype=bool)
                    if start_pos < end_pos:
                        not_in_recomb[start_pos:(end_pos+1)] = False
                    else:
                        not_in_recomb[start_pos:]   = False
                        not_in_recomb[:(end_pos+1)] = False
                    in_CF = np.logical_and(in_CF, not_in_recomb)

            non_clonal_k += 1
            k += 1

    sum_in_CF = np.sum(in_CF)
    return [t, recomb_num, k, sum_in_CF / L]

In [28]:
N = 20
rho = 1
delta = 20
L = 100

P1 = aff_P1(delta, L)
P2 = aff_P2(delta, L)
CF_P1 = CF_P(N, rho, P1)
CF_P2 = CF_P(N, rho, P2)

print(f"aff_P1: {P1}, aff_P2: {P2}")
print(f"CF_P1: {CF_P1}, CF_P2: {CF_P2}")

aff_P1: 0.16806722689075632, aff_P2: 0.19881589415593318
CF_P1: 0.5624462648770481, CF_P2: 0.5083280783071903


In [29]:
list_sim_results = []
for _ in tqdm(range(1_000_000)):
    sim_result = birth_death_recomb_sim(N, rho, delta, L)
    list_sim_results.append(sim_result)

100%|██████████| 1000000/1000000 [05:16<00:00, 3154.92it/s]


In [30]:
array_sim_results = np.array(list_sim_results)
array_sim_results

array([[1.28223319, 2.        , 1.        , 0.78      ],
       [1.94284655, 3.        , 1.        , 0.43      ],
       [2.14627728, 6.        , 1.        , 0.2       ],
       ...,
       [1.63112011, 8.        , 1.        , 0.25      ],
       [2.93499129, 7.        , 1.        , 0.11      ],
       [2.77281357, 4.        , 1.        , 0.16      ]],
      shape=(1000000, 4))

In [31]:
sim_CF_P = np.mean(array_sim_results[:, 3])
sim_CF_P

np.float64(0.47026040999999996)